# 🛍️ Shopping Mall Customer Segmentation
## Member 3 — DBSCAN Clustering (Deep Optimization)

### 📌 针对老师反馈的深度优化：
1. **彻底解决重叠问题**：使用 **t-SNE 提取的 3D 特征**进行聚类和可视化。t-SNE 能够将原本在原始空间中重叠的点在 3D 空间中彻底拉开，使不同簇像“新加坡人”和“马来西亚人”一样界限分明。
2. **噪声识别**：DBSCAN 能够识别出不属于任何簇的“噪声点”，这在商业上代表了极少数的异常客户。
3. **深化业务行动建议**：针对每个聚类结果提供具体的商业对策。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.metrics import v_measure_score
from mpl_toolkits.mplot3d import Axes3D

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## 1. Load Data
**目的**：加载在 Step 0 中生成的 t-SNE 特征。

In [ ]:
X_tsne = np.load('X_tsne.npy')
df_original = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')
print(f'✅ Loaded t-SNE features shape: {X_tsne.shape}')

## 2. Train & Evaluate
**目的**：在 t-SNE 空间进行密度聚类。

In [ ]:
# 使用 t-SNE 空间下的较优参数
dbscan = DBSCAN(eps=5.0, min_samples=10)
labels = dbscan.fit_predict(X_tsne)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

v_score = v_measure_score(df_original['Gender'], labels)
print(f"Clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")
print(f"V-measure Score: {v_score:.4f}")

## 3. 3D Visualization (No Overlapping)
**意义**：在 t-SNE 空间中，簇被彻底拉开，视觉上非常清晰。

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

mask_noise = (labels == -1)
mask_clusters = (labels != -1)

# 噪声点用灰色表示
ax.scatter(X_tsne[mask_noise, 0], X_tsne[mask_noise, 1], X_tsne[mask_noise, 2], 
           c='lightgrey', label='Noise', s=10, alpha=0.3)

# 聚类点
scatter = ax.scatter(X_tsne[mask_clusters, 0], X_tsne[mask_clusters, 1], X_tsne[mask_clusters, 2], 
                     c=labels[mask_clusters], cmap='tab10', s=20, alpha=0.6)

ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_zlabel('t-SNE 3')
ax.set_title('3D DBSCAN Clustering in t-SNE Space')
ax.legend()
plt.show()

print("""💡 业务洞察与行动建议：
1. 观察：DBSCAN 识别出了核心客户群和边缘客户（噪声）。
2. 行动：针对噪声点（异常客户），应检查其是否为潜在的欺诈账户或极高价值的特殊客户。
3. 行动：针对核心簇，应分析其共同特征，作为商场未来招商引资的参考依据。""")